# **Colab Install **

# **2️⃣ Imports**

In [ ]:
!pip install -q -U pypdf
!pip install -q -U sentence-transformers
!pip install -q -U pinecone
!pip install -q "google-auth==2.49.0"
!pip install -q -U tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 10.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.19.0 requires google-auth[requests]<3.0.0,>=2.56.0, but you have google-auth 2.49.0 which is incompatible.


In [ ]:
import os
import re
import json
import getpass

from pathlib import Path

from pypdf import PdfReader

from sentence_transformers import SentenceTransformer

from pinecone import Pinecone, ServerlessSpec

from google import genai

# **3️⃣ API Keys**

In [ ]:
import getpass

PINECONE_API_KEY = getpass.getpass("Enter Pinecone API Key:pcsk_22LgTp_AmBZ9uFzTGZyvzGQE8iTEQd8HrrT79zcBjbNQ2fgYw5A1AEbkYYVuRWaNRP27fp ")
GEMINI_API_KEY = getpass.getpass("Enter Gemini API Key:AQ.Ab8RN6IavU9FRfBQmVh2ke81DjvByx3lx5sHnSJyIzXDYD0pYA ")

Enter Pinecone API Key:pcsk_22LgTp_AmBZ9uFzTGZyvzGQE8iTEQd8HrrT79zcBjbNQ2fgYw5A1AEbkYYVuRWaNRP27fp ··········
Enter Gemini API Key:AQ.Ab8RN6IavU9FRfBQmVh2ke81DjvByx3lx5sHnSJyIzXDYD0pYA ··········


# **Initialize Pincone**

In [ ]:
pc = Pinecone(api_key=PINECONE_API_KEY)
print("Pinecone client initialized successfully.")

Pinecone client initialized successfully.


# **4️⃣ Initialize Gemini**

In [ ]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

MODEL_NAME = "gemini-3.6-flash"

print("Gemini initialized successfully.")

Gemini initialized successfully.


# **5️⃣ Download/Upload PDFs**

In [ ]:
from google.colab import files

uploaded = files.upload()

print("Uploaded files:")
for filename in uploaded.keys():
    print(filename)

Saving first Aid book.pdf to first Aid book.pdf
Saving IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf to IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
Uploaded files:
first Aid book.pdf
IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf


# **6️⃣ Create Medical Documents Folder**

In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/medical_documents")

DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(DATA_DIR)

/content/medical_documents


# **Move uploaded files:**

In [ ]:
import shutil
import os

for filename in uploaded.keys():

    source = Path("/content") / filename
    destination = DATA_DIR / filename

    # Check if the file with the name from uploaded.keys() exists
    if not os.path.exists(source):
        # If not, try the original name without the (1) suffix, assuming this is a common Colab upload renaming issue
        if " (1).pdf" in filename:
            original_filename = filename.replace(" (1).pdf", ".pdf")
            alternate_source = Path("/content") / original_filename
            if os.path.exists(alternate_source):
                source = alternate_source
                # Also update destination to match the actual filename being moved
                destination = DATA_DIR / original_filename
            else:
                print(f"Warning: Neither {source} nor {alternate_source} found. Skipping {filename}.")
                continue
        else:
            print(f"Warning: File {source} not found. Skipping {filename}.")
            continue

    shutil.move(str(source), str(destination))

print("Files moved to:", DATA_DIR)

for file in DATA_DIR.iterdir():
    print(file.name)

Files moved to: /content/medical_documents
IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
first Aid book.pdf


# **7️⃣ Extract PDF Text**

Now , Actual RAG start




In [ ]:
def extract_pdf_text(pdf_path):

    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):

        text = page.extract_text()

        if text:
            pages.append({
                "page": page_number,
                "text": text
            })

    return pages

# **8️⃣ Extract Both PDFs**

In [ ]:
all_pages = []

for pdf_file in DATA_DIR.glob("*.pdf"):

    print(f"Processing: {pdf_file.name}")

    pages = extract_pdf_text(pdf_file)

    for page in pages:

        page["source"] = pdf_file.name

        all_pages.append(page)

print("\nTotal pages extracted:", len(all_pages))

Processing: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
Processing: first Aid book.pdf

Total pages extracted: 831


# **Check:**

In [ ]:
for item in all_pages[:3]:

    print("=" * 80)

    print("SOURCE:", item["source"])
    print("PAGE:", item["page"])

    print(item["text"][:1000])

SOURCE: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
PAGE: 1
 
 
 
 
 
 
INTERNATIONAL FIRST AID, 
RESUSCITATION AND EDUCATION  
GUIDELINES 2025 
 
 
  

SOURCE: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
PAGE: 2
 
 
Contents 
TOPIC INDEX 4 
INTRODUCTION 11 
CONTEXTUALIZING FIRST AID AND FIRST AID EDUCATION 27 
FIRST AID EDUCATION 71 
FIRST AID 111 
GENERAL APPROACH 113 
RESUSCITATION 151 
BREATHING PROBLEMS 231 
TRAUMA 255 
MEDICAL CONDITIONS 357 
ENVIRONMENTAL 445 
MENTAL DISTRESS 485 
GLOSSARY AND REFERENCES 523 
 
 
 
 
Audience: First aid programme designers, programme managers, education and scienti ﬁc 
committees, trainers 
 
 
 
Red Cross Red Crescent Networks 
Coordinated by IFRC Global First Aid Reference Centre 
 
© International Federation of Red Cross and Red Crescent Societies, Geneva, 2025 
Copies of all or part of this study may be made for non-commercial use, providing the source is 
acknowledged. The 

# **9️⃣ Clean Text**

PDF extraction mein extra spaces/newlines aa sakte hain.

In [ ]:
def clean_text(text):

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\s+([,.!?;:])', r'\1', text)

    return text.strip()

# **Apply:**

In [ ]:
for item in all_pages:

    item["text"] = clean_text(item["text"])

print(all_pages[0]["text"][:1000])

INTERNATIONAL FIRST AID, RESUSCITATION AND EDUCATION GUIDELINES 2025


# **🔟 Chunking**

This is very important step.
We will not directly embedd our page **bold text**

In [ ]:
def chunk_text(text, chunk_size=800, overlap=150):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if chunk.strip():
            chunks.append(chunk.strip())

        start += chunk_size - overlap

    return chunks

# **1️⃣1️⃣ Create Chunks With Metadata**

In [ ]:
documents = []

chunk_id = 0

for page in all_pages:

    chunks = chunk_text(
        page["text"],
        chunk_size=800,
        overlap=150
    )

    for chunk in chunks:

        documents.append({
            "id": f"chunk-{chunk_id}",
            "text": chunk,
            "source": page["source"],
            "page": page["page"]
        })

        chunk_id += 1

print("Total chunks:", len(documents))

Total chunks: 3408


# **Check:**

In [ ]:
for doc in documents[:3]:

    print("=" * 80)

    print("ID:", doc["id"])
    print("Source:", doc["source"])
    print("Page:", doc["page"])

    print(doc["text"][:500])

ID: chunk-0
Source: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
Page: 1
INTERNATIONAL FIRST AID, RESUSCITATION AND EDUCATION GUIDELINES 2025
ID: chunk-1
Source: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
Page: 2
Contents TOPIC INDEX 4 INTRODUCTION 11 CONTEXTUALIZING FIRST AID AND FIRST AID EDUCATION 27 FIRST AID EDUCATION 71 FIRST AID 111 GENERAL APPROACH 113 RESUSCITATION 151 BREATHING PROBLEMS 231 TRAUMA 255 MEDICAL CONDITIONS 357 ENVIRONMENTAL 445 MENTAL DISTRESS 485 GLOSSARY AND REFERENCES 523 Audience: First aid programme designers, programme managers, education and scienti ﬁc committees, trainers Red Cross Red Crescent Networks Coordinated by IFRC Global First Aid Reference Centre © International 
ID: chunk-2
Source: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
Page: 2
source is acknowledged. The IFRC would appreciate receiving details of its use. Requests for commercial reproduc

# ** 1️⃣2️⃣ Hugging Face Embedding Model**

**Now the role of Hugging Face.**
**This model is available through Hugging Face's Sentence Transformers ecosystem.**

In [ ]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded.


# **1️⃣3️⃣ Generate Embeddings**
You should see something similar to:

Embedding shape: (number_of_chunks, 384)

because this model produces 384-dimensional embeddings. **bold text**

In [ ]:
texts = [
    doc["text"]
    for doc in documents
]

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32,
    normalize_embeddings=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/107 [00:00<?, ?it/s]

Embedding shape: (3408, 384)


# **1️⃣5️⃣ Create Pinecone Index**
Because all-MiniLM-L6-v2 produces 384-dimensional embeddings, our index dimension must match.

Pinecone's current documentation explicitly requires the dimension of a bring-your-own-vector index to match the embedding model's vector dimension bold text **bold text**

In [ ]:
INDEX_NAME = "guardianai-firstaid"

# Check if the index exists and delete it if it does to ensure correct dimension
if INDEX_NAME in [index["name"] for index in pc.list_indexes()]:
    print(f"Deleting existing index: {INDEX_NAME}")
    pc.delete_index(INDEX_NAME)
    print("Index deleted.")

# Create the index with the correct dimension
pc.create_index(
    name=INDEX_NAME,
    dimension=384, # Correct dimension for the embedding model
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

print("Index created successfully with dimension 384.")

Deleting existing index: guardianai-firstaid
Index deleted.
Index created successfully with dimension 384.


# **1️⃣5️⃣ Create Pinecone Index**

Because all-MiniLM-L6-v2 produces 384-dimensional embeddings, our index dimension must match.

Pinecone's current documentation explicitly requires the dimension of a bring-your-own-vector index to match the embedding model's vector dimension **bold text**

# **1️⃣6️⃣ Connect to Index**

In [ ]:
index = pc.Index(INDEX_NAME)

print(index.describe_index_stats())

DescribeIndexStatsResponse(dimension=384, total_vector_count=0, metric='cosine', namespaces=0)


# **1️⃣7️⃣ Upload Vectors to Pinecone**

In [ ]:
vectors = []

for i, doc in enumerate(documents):

    vector = {
        "id": doc["id"],
        "values": embeddings[i].tolist(),
        "metadata": {
            "text": doc["text"],
            "source": doc["source"],
            "page": doc["page"]
        }
    }

    vectors.append(vector)

# **Upsert in batches:**

In [ ]:
batch_size = 100

for i in range(0, len(vectors), batch_size):

    batch = vectors[i:i + batch_size]

    index.upsert(
        vectors=batch
    )

    print(
        f"Uploaded {i} - {i + len(batch)}"
    )

print("All vectors uploaded.")

Uploaded 0 - 100
Uploaded 100 - 200
Uploaded 200 - 300
Uploaded 300 - 400
Uploaded 400 - 500
Uploaded 500 - 600
Uploaded 600 - 700
Uploaded 700 - 800
Uploaded 800 - 900
Uploaded 900 - 1000
Uploaded 1000 - 1100
Uploaded 1100 - 1200
Uploaded 1200 - 1300
Uploaded 1300 - 1400
Uploaded 1400 - 1500
Uploaded 1500 - 1600
Uploaded 1600 - 1700
Uploaded 1700 - 1800
Uploaded 1800 - 1900
Uploaded 1900 - 2000
Uploaded 2000 - 2100
Uploaded 2100 - 2200
Uploaded 2200 - 2300
Uploaded 2300 - 2400
Uploaded 2400 - 2500
Uploaded 2500 - 2600
Uploaded 2600 - 2700
Uploaded 2700 - 2800
Uploaded 2800 - 2900
Uploaded 2900 - 3000
Uploaded 3000 - 3100
Uploaded 3100 - 3200
Uploaded 3200 - 3300
Uploaded 3300 - 3400
Uploaded 3400 - 3408
All vectors uploaded.


# **1️⃣8️⃣ Test Pinecone Retrieval**

Abhi Gemini ko involve nahi karna.

Pehle check karenge:

Pinecone correct information retrieve kar raha hai ya nahi **bold text**.

In [ ]:
def search_pinecone(query, top_k=5):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    results = index.query(
        vector=query_embedding,
        top_k=top_k,
        include_metadata=True
    )

    return results

# **Test:**

In [ ]:
query = "What should I do if someone has severe bleeding?"

results = search_pinecone(query)

for match in results["matches"]:

    print("=" * 80)

    print("Score:", match["score"])

    print("Source:", match["metadata"]["source"])

    print("Page:", match["metadata"]["page"])

    print("Text:")
    print(match["metadata"]["text"][:1000])

Score: 0.628287315
Source: IFRC International First Aid, Resuscitation and Education Guidelines 2025.pdf
Page: 257
Text:
● Action to stem the ﬂow of blood should be taken as soon as possible. Even a cupful of blood, although not immediately life-threatening, can lead to fast deterioration if not stopped early on. First aid steps 1. Ask the person to apply direct pressure to their injury with their hands. 2. Help the person to lie down. 3. Access EMS. 4. Apply direct pressure to the bleed. If blood soaks through the dressing, apply a second dressing over the ﬁrst one and apply greater pressure. 5. If direct pressure is ineffective and the person is bleeding from an arm or leg, consider applying a tourniquet or haemostatic dressing, if available. Continue to put direct pressure on the bleed. In the case that a tourniquet is ineffective, you can add a second tourniquet above the ﬁrst. 6. Shock is likely to deve
Score: 0.619432449
Source: IFRC International First Aid, Resuscitation and Edu

# **1️⃣9️⃣ Now Gemini + RAG**
**Now, We will ready the brain of  actual Knowlege base agent **

In [ ]:
def build_context(results):

    context_parts = []

    for match in results["matches"]:

        metadata = match["metadata"]

        context_parts.append(
            f"""
SOURCE: {metadata['source']}
PAGE: {metadata['page']}

CONTENT:
{metadata['text']}
"""
        )

    return "\n\n".join(context_parts)

# **2️⃣0️⃣ Generate Grounded Answer**

In [ ]:
def generate_first_aid_answer(
    user_query,
    top_k=5
):

    results = search_pinecone(
        user_query,
        top_k=top_k
    )

    context = build_context(results)

    prompt = f"""
You are the Knowledge Agent of GuardianAI.

GuardianAI is an AI-assisted emergency
coordination and first-response support system.

Your task is to provide grounded first-aid
information using ONLY the retrieved context.

Do NOT diagnose medical conditions.

Do NOT invent medical instructions.

If the retrieved context is insufficient,
clearly say that reliable information was
not found and recommend seeking professional
medical assistance.

User situation:
{user_query}

Retrieved medical knowledge:
{context}

Provide:

1. Immediate first-aid guidance
2. Important safety warnings
3. When professional emergency help is needed
4. Sources used

Keep the answer clear and actionable.
"""

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text, results

# **2️⃣1️⃣ Test the Knowledge Agent**

In [ ]:
answer, results = generate_first_aid_answer(
    "Someone has severe bleeding after a road accident."
)

print(answer)

Based on the provided medical guidelines, here is the clear and actionable first-aid information for a road accident involving severe bleeding:

### 1. Immediate First-Aid Guidance
* **Apply Direct Pressure:** Immediately apply direct pressure to the bleeding area to control the hemorrhage as quickly as possible. Early hemorrhage control is a central priority.
* **If Person Becomes Unresponsive:** After stopping the bleeding, if the injured person becomes unresponsive, open their airway and check for breathing.
* **Tourniquet Usage (Specific Scenarios):** In situations of disaster, conflict, or immediate danger zone evacuations, tourniquets may be used as a short-term measure to stop life-threatening bleeding. *Note: A tourniquet should only be released under the guidance of a medical professional.*

### 2. Important Safety Warnings
* **Safety First:** The safety and security of both you (the first-aid provider) and the injured person are paramount and take precedence over providing im

# **2️⃣2️⃣ Make a Reusable Knowledge Agent**

**Ab is function ko proper agent bana dete hain.**

In [ ]:
class KnowledgeAgent:

    def __init__(
        self,
        embedding_model,
        pinecone_index,
        gemini_client,
        model_name
    ):

        self.embedding_model = embedding_model
        self.index = pinecone_index
        self.client = gemini_client
        self.model_name = model_name

    def retrieve(
        self,
        query,
        top_k=5
    ):

        query_embedding = self.embedding_model.encode(
            query,
            normalize_embeddings=True
        ).tolist()

        results = self.index.query(
            vector=query_embedding,
            top_k=top_k,
            include_metadata=True
        )

        return results

    def answer(
        self,
        query,
        top_k=5
    ):

        results = self.retrieve(
            query,
            top_k
        )

        context = build_context(results)

        prompt = f"""
You are GuardianAI's Knowledge Agent.

Use ONLY the retrieved trusted context.

User:
{query}

Retrieved context:
{context}

Rules:

- Do not diagnose.
- Do not invent medical facts.
- Do not provide unsupported instructions.
- If information is insufficient, recommend professional help.
- Clearly identify the source material used.

Return concise, safety-focused guidance.
"""

        response = self.client.models.generate_content(
            model=self.model_name,
            contents=prompt
        )

        return {
            "query": query,
            "answer": response.text,
            "sources": [
                {
                    "source": match["metadata"]["source"],
                    "page": match["metadata"]["page"],
                    "score": match["score"]
                }
                for match in results["matches"]
            ]
        }

# **2️⃣3️⃣ Initialize Agent**

In [ ]:
knowledge_agent = KnowledgeAgent(
    embedding_model=embedding_model,
    pinecone_index=index,
    gemini_client=client,
    model_name=MODEL_NAME
)

# **2️⃣4️⃣ Final Test**

In [ ]:
result = knowledge_agent.answer(
    "My brother was injured in a road accident and is bleeding heavily."
)

print("ANSWER")
print("=" * 80)

print(result["answer"])

print("\nSOURCES")
print("=" * 80)

for source in result["sources"]:

    print(
        source["source"],
        "| Page:",
        source["page"],
        "| Score:",
        round(source["score"], 4)
    )

ANSWER
**Severe external bleeding is a life-threatening emergency. Access Emergency Medical Services (EMS) or arrange urgent transfer to a surgical unit immediately.**

Based on the provided medical guidelines, follow these immediate first aid steps to control heavy bleeding:

1. **Access EMS:** Call emergency services right away.
2. **Apply Direct Pressure:** Apply firm, direct pressure to the wound immediately using your hands or a dressing. If he is able, ask your brother to apply direct pressure to his injury with his hands while you prepare. 
3. **Position Him:** Help him to lie down.
4. **Manage Bleeding/Dressings:** 
   * Continue applying firm direct pressure. 
   * If blood soaks through the dressing, apply a second dressing directly over the first one and apply greater pressure. Do not remove the first dressing.
   * If the wound is large or gaping, deep wound packing may be applied.
5. **Use a Tourniquet (If Needed for Arms or Legs):** 
   * If direct pressure is ineffective

# **2️⃣5️⃣ Test Multiple Questions**

**This is very important**

In [ ]:
test_questions = [

    "What should I do for severe bleeding?",

    "What first aid is appropriate for a burn?",

    "What should I do if someone is choking?",

    "What should I do if someone is unconscious?",

    "What should I do after a serious road accident?",

    "What should I do for a suspected fracture?"
]


for question in test_questions:

    print("\n")
    print("=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    result = knowledge_agent.answer(question)

    print(result["answer"])

    print("\nSources:")

    for source in result["sources"][:3]:

        print(
            f"- {source['source']} "
            f"(page {source['page']}, "
            f"score={source['score']:.3f})"
        )



QUESTION: What should I do for severe bleeding?
Based on the provided context, here are the step-by-step first aid instructions for severe bleeding:

### First Aid Steps for Severe Bleeding

1. **Ask for Immediate Pressure:** Ask the injured person to apply direct pressure to their injury using their hands.
2. **Position the Person:** Help the person lie down.
3. **Access EMS:** Call Emergency Medical Services (EMS) immediately and prepare for urgent medical transfer. *(Note: Shock is likely to develop with severe bleeding).*
4. **Apply Direct Pressure:** 
   * Apply direct pressure to the bleed using a gloved hand, bandage, or sterile gauze. 
   * If the wound is large or gaping, deep wound packing can be applied.
   * If blood soaks through the initial dressing, **do not remove it**; place a second dressing over the first and apply greater pressure.
5. **Consider a Tourniquet or Haemostatic Dressing (For Limb Wounds):**
   * If direct pressure is ineffective for severe/uncontrolled